# Ejercicios 04 — Herencia y MRO

**Teoría:** [11_HerenciaMRO_REV.ipynb](../11_HerenciaMRO_REV.ipynb)

Aplicamos el **MRO** (Method Resolution Order) y la **herencia múltiple** a un caso real:
la **sucesión intestada** (fallecimiento sin testamento) según el Código Civil español.

> Modelo **didáctico**. Simplifica el derecho sucesorio para practicar POO; no sustituye asesoramiento jurídico.


## Caso real: fallecimiento de Don Juan García

**Situación:** Don Juan fallece **sin testamento** (sucesión *ab intestato*).

**Parientes vivos en el momento del óbito:**

| Persona | Relación con Don Juan | Notas |
|---------|----------------------|-------|
| Ana García | Hija | Descendiente directa |
| Carlos García | Hijo | Descendiente directo |
| Luis García | Nieto | Hijo de Pedro (hijo de Don Juan **premuerto**) → *representación* |
| Miguel García | Hermano | Colateral |
| Elena Ruiz | Cónyuge viuda | Derechos legitimarios / usufructo (simplificado) |
| Rosa Pérez | Madre | Ascendiente (padre ya fallecido) |

**Pregunta:** ¿Quién hereda y con qué **prioridad**? ¿Cómo modelamos roles múltiples con MRO?


## 1. Marco legal (simplificado)

Orden de llamamiento en sucesión intestada (Código Civil, aproximación pedagógica):

1. **Descendientes** (hijos; nietos por representación) — arts. 930 y ss.
2. **Ascendientes** (padres, abuelos) — art. 936
3. **Cónyuge** viudo — arts. 944 y ss. (posición simplificada en el modelo)
4. **Colaterales** (hermanos, sobrinos…) — arts. 944 y ss.
5. **Estado** — herencia vacante


In [ ]:
# Constantes pedagógicas: prioridad (menor = más preferente) y artículo de referencia
PRIORIDAD_LEGAL = {
    "Descendiente": {"prioridad": 1, "articulo": "930", "tipo": "1º — descendientes"},
    "Ascendiente":  {"prioridad": 2, "articulo": "936", "tipo": "2º — ascendientes"},
    "Conyuge":      {"prioridad": 3, "articulo": "944", "tipo": "3º — cónyuge (simplificado)"},
    "Colateral":    {"prioridad": 4, "articulo": "944+", "tipo": "4º — colaterales"},
    "Estado":       {"prioridad": 5, "articulo": "956", "tipo": "herencia vacante"},
}

for rol, datos in PRIORIDAD_LEGAL.items():
    print(f"{datos['prioridad']}º  {rol:12}  art. {datos['articulo']:4}  →  {datos['tipo']}")

## 2. Modelo de clases con herencia múltiple

Jerarquía base:

- `Heredero` — clase raíz (cualquier llamado a la herencia)
- Categorías: `Descendiente`, `Ascendiente`, `Conyuge`, `Colateral`, `Estado`
- Roles concretos: `Hijo`, `Nieto`, `Padre`, `Hermano`, …

Cada clase define `reclamar()`; el **MRO** decide cuál se ejecuta cuando hay herencia múltiple.


In [ ]:
class Heredero:
    """Base de cualquier persona llamada a suceder."""
    prioridad = 99
    articulo = "-"
    tipo_juridico = "Heredero (base)"

    def __init__(self, nombre):
        self.nombre = nombre

    def reclamar(self):
        return f"{self.nombre}: reclamación genérica de heredero"

    def ficha(self):
        return (
            f"{self.nombre:15} | prioridad={self.prioridad} | "
            f"art.{self.articulo:4} | {self.tipo_juridico}"
        )


class Descendiente(Heredero):
    prioridad = 1
    articulo = "930"
    tipo_juridico = "Descendiente"

    def reclamar(self):
        return f"{self.nombre}: hereda como DESCENDIENTE (art. {self.articulo})"


class Ascendiente(Heredero):
    prioridad = 2
    articulo = "936"
    tipo_juridico = "Ascendiente"

    def reclamar(self):
        return f"{self.nombre}: hereda como ASCENDIENTE (art. {self.articulo})"


class Conyuge(Heredero):
    prioridad = 3
    articulo = "944"
    tipo_juridico = "Cónyuge"

    def reclamar(self):
        return f"{self.nombre}: derechos como CÓNYUGE (art. {self.articulo}, simplificado)"


class Colateral(Heredero):
    prioridad = 4
    articulo = "944+"
    tipo_juridico = "Colateral"

    def reclamar(self):
        return f"{self.nombre}: hereda como COLATERAL (art. {self.articulo})"


class Estado(Heredero):
    prioridad = 5
    articulo = "956"
    tipo_juridico = "Estado / vacante"

    def reclamar(self):
        return "El ESTADO sucede: herencia vacante (art. 956)"


# Roles concretos
class Hijo(Descendiente):
    tipo_juridico = "Hijo (descendiente directo)"

    def reclamar(self):
        return f"{self.nombre}: HIJO — máxima preferencia entre descendientes (art. 930)"


class Nieto(Descendiente):
    """Nieto que hereda por representación del padre/madre premuerto."""
    tipo_juridico = "Nieto (representación)"

    def reclamar(self):
        return f"{self.nombre}: NIETO por REPRESENTACIÓN (ocupa el lugar del progenitor premuerto)"


class Padre(Ascendiente):
    tipo_juridico = "Padre/Madre (ascendiente)"

    def reclamar(self):
        return f"{self.nombre}: ASCENDIENTE en línea directa (art. 936)"


class Hermano(Colateral):
    tipo_juridico = "Hermano (colateral)"

    def reclamar(self):
        return f"{self.nombre}: HERMANO — colateral preferente (arts. 944 y ss.)"


print("Jerarquía creada.")
print("MRO de Hijo:", [c.__name__ for c in Hijo.mro()])
print("MRO de Nieto:", [c.__name__ for c in Nieto.mro()])
print("MRO de Hermano:", [c.__name__ for c in Hermano.mro()])

## 3. Caso complejo: heredero con múltiples relaciones

Supongamos un caso pedagógico (adopción / parentesco complejo) donde **una misma persona**
es a la vez **hijo** y **hermano** del causante respecto de distintas líneas familiares.

Modelamos:

```python
class HerederoComplejo(Hijo, Hermano):
    pass
```

El **MRO** decide qué `reclamar()` prevalece: primero `Hijo`, luego `Hermano`, etc.


In [ ]:
class HerederoComplejo(Hijo, Hermano):
    """Persona con roles concurrentes: hijo Y hermano (caso didáctico)."""
    tipo_juridico = "Heredero con roles múltiples"


# Cálculo del MRO
mro_nombres = [c.__name__ for c in HerederoComplejo.mro()]
print("MRO de HerederoComplejo:")
for i, nombre in enumerate(mro_nombres, start=1):
    print(f"  {i}. {nombre}")

print()
print("__mro__ (tupla):", HerederoComplejo.__mro__)
print("mro()   (lista):", HerederoComplejo.mro())

### Resolución paso a paso (C3)

Para `HerederoComplejo(Hijo, Hermano)`:

1. Empieza por la propia clase: `HerederoComplejo`
2. Continúa por el primer padre: `Hijo` → `Descendiente` …
3. Luego el segundo padre: `Hermano` → `Colateral` …
4. Fusiona sin repetir clases (`C3 linearization`)
5. Resultado típico:  
   `HerederoComplejo → Hijo → Descendiente → Hermano → Colateral → Heredero → object`

Así, al llamar `reclamar()`, Python encuentra primero el método de **`Hijo`** (prioridad jurídica 1ª).


In [ ]:
persona = HerederoComplejo("Alex García")
print(persona.reclamar())
print(persona.ficha())
print()
print("¿Es instancia de Hijo?", isinstance(persona, Hijo))
print("¿Es instancia de Hermano?", isinstance(persona, Hermano))
print("¿Es instancia de Descendiente?", isinstance(persona, Descendiente))
print("¿Es instancia de Colateral?", isinstance(persona, Colateral))

## 4. Simulación del caso real

Construimos el inventario de parientes vivos y seleccionamos a quiénes llama la ley
según la **mejor prioridad** presente (modelo simplificado: si hay descendientes,
no se abre el llamamiento a ascendientes/colaterales para la nuda propiedad).


In [ ]:
class SucesionIntestada:
    """Simulador pedagógico de llamamiento ab intestato."""

    def __init__(self, causante):
        self.causante = causante
        self.parientes = []

    def añadir(self, persona: Heredero):
        self.parientes.append(persona)

    def mejor_prioridad(self):
        if not self.parientes:
            return None
        return min(p.prioridad for p in self.parientes)

    def llamados(self):
        """Parientes del mejor grado presente (misma prioridad)."""
        mejor = self.mejor_prioridad()
        if mejor is None:
            return []
        return [p for p in self.parientes if p.prioridad == mejor]

    def informe(self):
        print("=" * 72)
        print(f"SUCESIÓN INTESTADA de {self.causante}")
        print("=" * 72)
        print("\nParientes vivos (ficha MRO / rol):")
        for p in sorted(self.parientes, key=lambda x: (x.prioridad, x.nombre)):
            print(" -", p.ficha())

        llamados = self.llamados()
        print("\n>> Llamados a heredar (mejor prioridad presente):")
        if not llamados:
            print(" - (ninguno) → se abriría herencia vacante")
        for p in llamados:
            print(" -", p.reclamar())
        print("=" * 72)


sucesion = SucesionIntestada("Don Juan García")
sucesion.añadir(Hijo("Ana García"))
sucesion.añadir(Hijo("Carlos García"))
sucesion.añadir(Nieto("Luis García"))          # representación
sucesion.añadir(Hermano("Miguel García"))
sucesion.añadir(Conyuge("Elena Ruiz"))
sucesion.añadir(Padre("Rosa Pérez"))           # madre

sucesion.informe()

### Lectura jurídica (simplificada)

Hay **descendientes** vivos (Ana, Carlos, Luis) → prioridad **1**.

- Ana y Carlos heredan por derecho propio.
- Luis hereda por **representación** de Pedro (premuerto).
- Rosa (ascendiente), Miguel (colateral) y el cónyuge **no** desplazan a los descendientes
  en este modelo simplificado de nuda propiedad (el cónyuge puede conservar derechos
  legitimarios/usufructo que aquí no cuantificamos).


## 5. Caso más complejo: líneas paterna y materna

Modelamos dos líneas de ascendientes. El MRO de una clase que hereda de ambas
ilustra el orden **izquierda → derecha** de la declaración de bases.


In [ ]:
class LineaPaterna(Ascendiente):
    tipo_juridico = "Línea paterna"

    def reclamar(self):
        return f"{self.nombre}: reclamación por LÍNEA PATERNA"


class LineaMaterna(Ascendiente):
    tipo_juridico = "Línea materna"

    def reclamar(self):
        return f"{self.nombre}: reclamación por LÍNEA MATERNA"


class AscendienteBilineal(LineaPaterna, LineaMaterna):
    """Ascendiente con vínculos en ambas líneas (didáctico)."""
    tipo_juridico = "Ascendiente bilineal"


print("MRO AscendienteBilineal:", [c.__name__ for c in AscendienteBilineal.mro()])
abuelo = AscendienteBilineal("Tomás García")
print(abuelo.reclamar())
print(abuelo.ficha())

# Si invertimos el orden de bases, cambia el MRO y el método elegido
class AscendienteBilinealInvertido(LineaMaterna, LineaPaterna):
    tipo_juridico = "Ascendiente bilineal (orden invertido)"

print()
print("MRO invertido:", [c.__name__ for c in AscendienteBilinealInvertido.mro()])
print(AscendienteBilinealInvertido("Tomás García").reclamar())

## 6. Caso extremo: herencia vacante (sin herederos)


In [ ]:
vacante = SucesionIntestada("Doña Carmen López")
# Sin parientes añadidos
vacante.informe()

# El Estado sucede
estado = Estado("Administración General del Estado")
vacante.añadir(estado)
vacante.informe()

## 7. Caso real: adopción de mayor de edad

**Situación:** Doña Marta Ortega, **soltera**, sin hijos biológicos.  
Adopta a un **hombre mayor de edad** cumpliendo los requisitos legales de la adopción de mayores.

**Familia de Doña Marta en el momento que estudiamos:**

| Persona | Relación | Estado | Notas |
|---------|----------|--------|-------|
| Antonio Ortega | Padre | Vivo | Ascendiente |
| Carmen López | Madre | Viva | Ascendiente |
| Andrés Ortega | Hermano | Vivo | Colateral |
| Pablo Ortega | Hermano | **Premuerto** | Sus hijos heredan por *representación* (si llega el llamamiento colateral) |
| Lucía Ortega | Hermana | Viva | Colateral |
| Sofía Ortega | Sobrina | Viva | Hija de Andrés (hermano **vivo**) → no representa mientras Andrés viva |
| Diego Ortega | Sobrino | Vivo | Hijo de Pablo (**premuerto**) → representación del tío |
| Nora Ortega | Sobrina | Viva | Hija de Pablo (**premuerto**) → representación |
| Marcos Vidal | Adoptado | Vivo | Mayor de edad adoptado por Marta |

**Pregunta clave:** ¿cambia el llamamiento *ab intestato* **antes** y **después** de la adopción?

> En el modelo didáctico, el adoptado de mayor de edad pasa a ser **descendiente** de la adoptante
> (arts. 175 y ss. / efectos sucesorios asimilados al hijo). Simplificación pedagógica.


In [ ]:
class Sobrino(Colateral):
    """Sobrino/a: colateral; puede heredar por representación del hermano premuerto."""
    tipo_juridico = "Sobrino/a (colateral)"

    def __init__(self, nombre, por_representacion=False, de_quien=None):
        super().__init__(nombre)
        self.por_representacion = por_representacion
        self.de_quien = de_quien

    def reclamar(self):
        if self.por_representacion:
            return (
                f"{self.nombre}: SOBRINO/A por REPRESENTACIÓN de {self.de_quien} "
                f"(colateral, art. {self.articulo})"
            )
        return (
            f"{self.nombre}: SOBRINO/A de hermano vivo — "
            f"no desplaza a su progenitor en el llamamiento colateral"
        )


class HijoAdoptivo(Hijo):
    """Hijo por adopción (incl. adopción de mayor de edad, modelo didáctico)."""
    tipo_juridico = "Hijo adoptivo (mayor de edad)"

    def reclamar(self):
        return (
            f"{self.nombre}: HIJO ADOPTIVO (mayor de edad) — "
            f"descendiente de la adoptante (art. 930 / efectos de la adopción)"
        )


print("MRO de HijoAdoptivo:", [c.__name__ for c in HijoAdoptivo.mro()])
print("MRO de Sobrino:", [c.__name__ for c in Sobrino.mro()])

### 7.1 Antes de la adopción

Sin descendientes: el llamamiento cae en **ascendientes** (padres vivos).  
Hermanos y sobrinos quedan pospuestos mientras existan padres.


In [ ]:
def familia_marta(incluir_adoptado=False):
    """Construye el inventario de parientes de Doña Marta."""
    s = SucesionIntestada("Doña Marta Ortega")
    s.añadir(Padre("Antonio Ortega"))
    s.añadir(Padre("Carmen López"))  # Madre reutiliza clase Padre (ascendiente)
    s.añadir(Hermano("Andrés Ortega"))
    s.añadir(Hermano("Lucía Ortega"))
    # Pablo premuerto: no se añade como heredero vivo
    s.añadir(Sobrino("Sofía Ortega", por_representacion=False, de_quien="Andrés Ortega"))
    s.añadir(Sobrino("Diego Ortega", por_representacion=True, de_quien="Pablo Ortega"))
    s.añadir(Sobrino("Nora Ortega", por_representacion=True, de_quien="Pablo Ortega"))
    if incluir_adoptado:
        s.añadir(HijoAdoptivo("Marcos Vidal"))
    return s


print(">>> ESCENARIO A — ANTES de la adopción\n")
antes = familia_marta(incluir_adoptado=False)
antes.informe()

### 7.2 Después de la adopción

Marcos, adoptado mayor de edad, es **descendiente**.  
Prioridad 1 → desplaza a padres, hermanos y sobrinos en el llamamiento a la herencia
(modelo simplificado de nuda propiedad).


In [ ]:
print(">>> ESCENARIO B — DESPUÉS de la adopción\n")
despues = familia_marta(incluir_adoptado=True)
despues.informe()

print("\nComparación de llamados:")
print("  Antes :", [p.nombre for p in antes.llamados()])
print("  Después:", [p.nombre for p in despues.llamados()])

### 7.3 ¿Y si no hubiera padres? (solo colaterales)

Hipótesis: padres ya fallecidos, **sin** adopción. Entonces entran los **colaterales**:

- Hermanos vivos (Andrés, Lucía) por derecho propio
- Sobrinos de hermano **premuerto** (Diego, Nora) por **representación** de Pablo
- Sofía (hija de Andrés vivo) **no** es llamada mientras Andrés viva

Así se ve la diferencia entre sobrino “en vida del padre” y sobrino por representación.


In [ ]:
print(">>> ESCENARIO C — Sin padres ni adopción (solo colaterales)\n")
solo_colaterales = SucesionIntestada("Doña Marta Ortega")
solo_colaterales.añadir(Hermano("Andrés Ortega"))
solo_colaterales.añadir(Hermano("Lucía Ortega"))
solo_colaterales.añadir(Sobrino("Sofía Ortega", por_representacion=False, de_quien="Andrés Ortega"))
solo_colaterales.añadir(Sobrino("Diego Ortega", por_representacion=True, de_quien="Pablo Ortega"))
solo_colaterales.añadir(Sobrino("Nora Ortega", por_representacion=True, de_quien="Pablo Ortega"))
solo_colaterales.informe()

print("\nMatiz pedagógico sobre sobrinos:")
for p in solo_colaterales.parientes:
    if isinstance(p, Sobrino):
        marca = "LLAMADO" if p in solo_colaterales.llamados() and p.por_representacion else (
            "misma prioridad pero sin derecho propio mientras vive el padre"
            if not p.por_representacion else "LLAMADO por representación"
        )
        # En nuestro modelo simplificado TODOS los colaterales comparten prioridad 4;
        # filtramos "efectivos" como hermanos + sobrinos por representación.
        pass

# Llamados "efectivos" en sentido jurídico simplificado
efectivos = [
    p for p in solo_colaterales.parientes
    if isinstance(p, Hermano)
    or (isinstance(p, Sobrino) and p.por_representacion)
]
print("Colaterales con vocación efectiva (hermanos + sobrinos por representación):")
for p in efectivos:
    print(" -", p.reclamar())
print("Excluida mientras vive su padre:", "Sofía Ortega")

### 7.4 Lectura con MRO

`HijoAdoptivo` reutiliza la cadena de `Hijo` → `Descendiente` → `Heredero`:

```
HijoAdoptivo → Hijo → Descendiente → Heredero → object
```

Por eso, en el escenario B, `reclamar()` y la prioridad (= 1) se resuelven como en cualquier hijo,
aunque el vínculo sea adoptivo y el adoptado sea mayor de edad.

**Idea fuerza:** un solo cambio en el grafo de clases (añadir un descendiente) reordena por completo
quién aparece en `llamados()`, sin reescribir la lógica del simulador.


## 8. Diagrama del MRO (caso con roles múltiples)

```
HerederoComplejo
      │
      ├─► Hijo ──► Descendiente ─┐
      │                          ├─► Heredero ──► object
      └─► Hermano ──► Colateral ─┘
```

Orden efectivo (C3):  
`HerederoComplejo → Hijo → Descendiente → Hermano → Colateral → Heredero → object`


## 9. Tabla de prioridades jurídicas vs MRO

| Posición MRO (ej. `Hijo`) | Clase | Tipo jurídico | Prioridad legal | Art. CC |
|---------------------------|-------|---------------|-----------------|---------|
| 1 | Hijo | Descendiente directo | 1º (máxima) | 930 |
| 2 | Descendiente | Categoría | 1º | 930 |
| 3 | Heredero | Base | — | — |
| 4 | object | built-in | — | — |

Para `HerederoComplejo`, el MRO inserta también `Hermano` / `Colateral` **después**
de la rama de descendiente, de modo que el método de hijo prevalece.


In [ ]:
# Tabla dinámica a partir del MRO de HerederoComplejo
print(f"{'#':<4}{'Clase':<22}{'Tipo jurídico':<32}{'Prioridad':<10}{'Art.'}")
print("-" * 80)
for i, cls in enumerate(HerederoComplejo.mro(), start=1):
    if cls is object:
        print(f"{i:<4}{'object':<22}{'built-in':<32}{'-':<10}{'-'}")
        continue
    prio = getattr(cls, "prioridad", "-")
    art = getattr(cls, "articulo", "-")
    tipo = getattr(cls, "tipo_juridico", cls.__name__)
    print(f"{i:<4}{cls.__name__:<22}{tipo:<32}{str(prio):<10}{art}")

## 10. Conclusión: ¿por qué el MRO es útil aquí?

1. **Resuelve conflictos de roles:** si alguien es hijo *y* hermano, el MRO fija qué relación prevalece en la búsqueda de métodos.
2. **Modela la representación:** el nieto puede ser una subclase de `Descendiente` con lógica propia.
3. **Ordena líneas de herencia:** el orden de bases refleja preferencias (paterna/materna, etc.).
4. **Permite casos complejos** sin `if/elif` gigantes: la jerarquía expresa el dominio.

**Extensiones posibles:** legados, tercios (legítima / mejora / libre disposición),
desheredación, indignidad, aceptación y repudiación de herencia.


## 11. Tu turno — ejercicios prácticos

Resuelve en las celdas vacías. Puedes reutilizar las clases definidas arriba (re-ejecuta las celdas previas si hace falta).


### Ejercicio A

Crea la clase `Sobrino(Colateral)` con un `reclamar()` propio.  
Añádelo a una sucesión **sin** descendientes ni ascendientes ni cónyuge, solo con un hermano y un sobrino.  
¿Quién es llamado? ¿Por qué tienen la misma o distinta prioridad?


In [ ]:
# Escribe tu solución aquí


### Ejercicio B

Define `class DobleVinculo(Nieto, Hermano)` y muestra su MRO.  
Instancia un objeto y llama a `reclamar()`. ¿Qué método se ejecuta? Justifica con el MRO.


In [ ]:
# Escribe tu solución aquí


### Ejercicio C

Modifica el orden de herencia a `class DobleVinculoInvertido(Hermano, Nieto)` y compara el MRO y el resultado de `reclamar()` con el ejercicio B.


In [ ]:
# Escribe tu solución aquí


### Ejercicio D (reto)

Implementa `repartir_cuotas(llamados)` que, dado el resultado de `sucesion.llamados()`,
devuelva un diccionario `{nombre: cuota}` repartiendo el caudal **a partes iguales**
entre los llamados (modelo muy simplificado, sin estirpes).
Pruébalo con el caso de Don Juan.


In [ ]:
# Escribe tu solución aquí


### Ejercicio E — Adopción de mayor de edad

1. Replica el escenario A (antes) y B (después) con **tus propios nombres**.
2. Añade un segundo adoptado mayor de edad y observa `llamados()`.
3. ¿Qué ocurre si, además del adoptado, existiera un hijo biológico? ¿Comparte cuota en nuestro modelo?


In [ ]:
# Escribe tu solución aquí
